# AI工学101 — 第37回

## 回帰モデルを本気で比較する：Linear → Random Forest → Gradient Boosting

よしレベル、今日は **「同じデータを、複数のモデルに公平に戦わせる」** 回だ。💪

ここまで僕らは個別のモデルを学んできたけど、実務では次のような問いが重要になる。

> Linear RegressionとRandom Forest、どっちがいい？

この問いに対して、

```text
Random Forestの方が強そう
```

では答えにならない。

今日からは、

```text
同じデータ
同じ評価指標
同じCV
同じ前処理条件
```

で比較する。

そして重要なのは、

> **「複雑なモデルほど強い」わけではない。**

むしろ、

```text
データ量
ノイズ
特徴量
問題の構造
評価方法
計算コスト
説明可能性
```

との相性で勝者が変わる。

今日は**モデル選択を実験として設計する**ぞ。

---

# 🎯 今日のゴール

* Baselineを作れる
* `LinearRegression` を比較対象として使える
* `Ridge` / `Lasso` の役割を理解する
* `RandomForestRegressor` を使える
* `GradientBoostingRegressor` を使える
* MAE / RMSE / R² を比較できる
* Cross Validationで公平に比較できる
* 複雑なモデルが勝つとは限らない理由を説明できる
* ハイパーパラメータ調整の入口を理解する

---

# 📖 講義：約20〜25分

## 1. 回帰問題の復習

分類では、

```text
犬 / 猫
購入する / しない
```

のようなカテゴリを予測した。

回帰では、

```text
価格
売上
温度
需要
```

などの連続値を予測する。

例えば、

```text
家の特徴
↓
家の価格
```

。

---

# 🧠 2. Baselineを作る

まずモデルを作る前に、

> **「何もしなくてもどのくらいの性能か？」**

を確認する。

例えば、

```text
訓練データの平均価格
```

をすべての家の予測値にする。

これは、

```text
DummyRegressor
```

で作れる。

```python
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(
    strategy="mean"
)
```

ここで重要なのは、

> **AIモデルはBaselineを超えて初めて価値が出る。**

---

# 💻 実習1：データを用意する

今回は `scikit-learn` にある回帰データセットを使う。

```python
from sklearn.datasets import load_diabetes

data = load_diabetes(
    as_frame=True
)

X = data.data
y = data.target

print(X.shape)
print(y.shape)
```

特徴量を確認。

```python
print(X.head())
```

---

# 💻 実習2：Train / Test Split

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

ここでのポイント。

```text
test
```

は最終評価用。

モデル選択をするときに、

```text
何度もtestを見る
```

と、

> **モデル選択の過程でtestに適応してしまう。**

そこで次にCVを使う。

---

# 🧠 3. 評価指標

## MAE

Mean Absolute Error。

```text
|予測 - 正解|
```

の平均。

```python
from sklearn.metrics import mean_absolute_error
```

。

解釈しやすい。

---

## RMSE

Root Mean Squared Error。

大きな誤差をより強く罰する。

```python
from sklearn.metrics import mean_squared_error

rmse = mean_squared_error(
    y_true,
    y_pred,
    squared=False
)
```

ただし現在のscikit-learnでは、環境によっては次の書き方も使える。

```python
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(
    y_true,
    y_pred
)
```

---

## R²

```text
1
↓
非常に良い

0
↓
Baseline程度

負
↓
Baselineより悪い
```

。

---

# 💻 実習3：Baseline

```python
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

baseline = DummyRegressor(
    strategy="mean"
)

baseline.fit(
    X_train,
    y_train
)

pred = baseline.predict(
    X_test
)

print(
    "MAE:",
    mean_absolute_error(
        y_test,
        pred
    )
)

print(
    "RMSE:",
    root_mean_squared_error(
        y_test,
        pred
    )
)

print(
    "R2:",
    r2_score(
        y_test,
        pred
    )
)
```

まずこれを基準にする。

---

# 🧠 4. Linear Regression

次。

```python
from sklearn.linear_model import LinearRegression
```

。

```python
linear = LinearRegression()

linear.fit(
    X_train,
    y_train
)

pred = linear.predict(
    X_test
)
```

Linear Regressionは、

> **特徴量と目的変数の線形関係**

を仮定する。

概念的には、

```text
y = w1x1 + w2x2 + ... + b
```

。

---

# 🧠 5. RidgeとLasso

Linear Regressionの発展。

---

## Ridge

```python
from sklearn.linear_model import Ridge
```

。

損失に、

```text
重みの大きさへのペナルティ
```

を加える。

```text
MSE
+
L2正則化
```

。

```python
ridge = Ridge(
    alpha=1.0
)
```

。

`alpha` を大きくすると正則化が強くなる。

---

## Lasso

```python
from sklearn.linear_model import Lasso
```

。

こちらは、

```text
L1正則化
```

。

特徴として、

> **一部の係数を0にできる**

。

つまり、

```text
特徴量選択
```

のような効果を持つ。

---

# 💻 実習4：線形モデル比較

```python
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)
```

```python
models = {
    "Linear": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "Lasso": Lasso(
        alpha=1.0,
        max_iter=10000
    )
}
```

ループ。

```python
for name, model in models.items():

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        pred
    )

    print(
        name,
        mae
    )
```

---

# 🧠 6. Random Forest Regressor

次。

```python
from sklearn.ensemble import RandomForestRegressor
```

。

Random Forestは、

```text
Decision Tree
```

を大量に作って、

```text
平均
```

する。

概念的には、

```text
Tree 1
Tree 2
Tree 3
...
Tree 100
↓
平均
↓
予測
```

。

線形関係を仮定しない。

だから、

```text
非線形
```

な関係も扱える。

---

# 💻 実習5：Random Forest

```python
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
```

学習。

```python
rf.fit(
    X_train,
    y_train
)

pred = rf.predict(
    X_test
)
```

評価。

```python
print(
    mean_absolute_error(
        y_test,
        pred
    )
)
```

---

# 🧠 7. Gradient Boosting

次。

```python
GradientBoostingRegressor
```

。

Random Forestが、

```text
たくさんの木
↓
並列的
↓
平均
```

なのに対して、

Gradient Boostingは、

```text
最初のモデル
↓
残った誤差を学ぶモデル
↓
さらに残った誤差
↓
さらに学ぶ
```

。

概念的には、

```text
Model 1
↓
残差
↓
Model 2
↓
残差
↓
Model 3
```

。

---

# 💻 実習6：Gradient Boosting

```python
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
```

```python
gbr.fit(
    X_train,
    y_train
)

pred = gbr.predict(
    X_test
)
```

評価。

```python
print(
    mean_absolute_error(
        y_test,
        pred
    )
)
```

---

# 🧠 8. 学習率と木の数

Gradient Boostingでは、

```text
learning_rate
```

と、

```text
n_estimators
```

の関係が重要。

例えば、

```text
learning_rate 小
↓
1回の更新が慎重
↓
多くのモデルが必要
```

。

```text
learning_rate 大
↓
速く学習
↓
過学習の危険
```

。

---

# 🧠 9. でもTestだけでモデルを比較しない

ここが今日の核心。

例えば、

```text
Linear
MAE = 45

Random Forest
MAE = 43

Gradient Boosting
MAE = 42
```

。

Testデータ1回だけでは、

```text
偶然
```

の可能性がある。

そこで、

```text
Cross Validation
```

。

---

# 💻 実習7：Cross Validation

```python
from sklearn.model_selection import cross_val_score
```

。

今回は、

```text
5-fold CV
```

。

```python
from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

モデル比較。

```python
models = {
    "Linear": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}
```

評価。

```python
for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    mae = -scores

    print(
        name
    )

    print(
        "mean:",
        mae.mean()
    )

    print(
        "std:",
        mae.std()
    )
```

---

# 🧠 10. 平均だけ見ない

例えば、

```text
Model A
MAE mean = 40
std = 2

Model B
MAE mean = 39
std = 10
```

。

Bの平均は良い。

でも結果がかなり不安定。

つまり、

> **平均性能だけではなく、安定性も見る。**

---

# 💻 実習8：比較表を作る

```python
results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    mae = -scores

    results.append({
        "model": name,
        "MAE mean": mae.mean(),
        "MAE std": mae.std()
    })
```

DataFrame。

```python
results_df = pd.DataFrame(
    results
)

print(
    results_df.sort_values(
        "MAE mean"
    )
)
```

これで、

```text
モデル
↓
平均MAE
↓
ばらつき
```

を比較できる。

---

# 🧠 11. 「複雑なモデルが勝つ」とは限らない

なぜか。

---

## データが少ない

複雑なモデルは、

```text
訓練データの偶然のパターン
```

まで学習しやすい。

つまり、

```text
過学習
```

。

---

## 問題がほぼ線形

例えば、

```text
y = 3x + 5
```

。

この場合、

```text
巨大なRandom Forest
```

は必要ない。

Linear Regressionで十分。

---

## ノイズが大きい

複雑なモデルは、

```text
信号
```

だけでなく、

```text
ノイズ
```

まで拾う可能性がある。

---

## 説明可能性

例えば、

```text
銀行
医療
行政
```

。

場合によっては、

```text
性能がほんの少し高い
```

より、

```text
説明しやすい
```

ことの方が重要。

---

# 🧠 12. ハイパーパラメータとは？

例えば、

```python
RandomForestRegressor(
    n_estimators=300,
    max_depth=10
)
```

。

この、

```text
n_estimators
max_depth
```

は、

```text
ハイパーパラメータ
```

。

モデルが学習する値ではなく、

> **人間が学習前に設定する値**

。

---

# 💻 実習9：Random Forestのmax_depth

```python
for depth in [
    2,
    5,
    10,
    None
]:

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=depth,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    print(
        depth,
        -scores.mean()
    )
```

見る。

```text
浅い木
↓
単純
↓
Underfittingの可能性

深い木
↓
複雑
↓
Overfittingの可能性
```

。

---

# 🚨 ここで重要

Testデータを使って、

```text
max_depth
```

を何十回も調整しない。

正しい流れ。

```text
Train
↓
CV
↓
モデル選択
ハイパーパラメータ選択
↓
最後にTest
```

。

---

# 👾 ボス戦

## 「最高性能のモデル」を選べ

次の結果。

```text
Model A
MAE = 45
std = 1

Model B
MAE = 43
std = 2

Model C
MAE = 42
std = 8
```

どれを選ぶ？

答えは、

> **問題による。**

例えば、

```text
安全性が重要
```

なら、

```text
AやB
```

が候補になる。

平均性能が最重要なら、

```text
C
```

。

ただし、

```text
C
```

は、

> データ分割によって大きく性能が変わる可能性がある。

ここで重要なのは、

> **「一番数字が良い」だけで選ばない。**

---

# ✍️ 演習

## 問1

Baselineを作る理由を説明してください。

---

## 問2

次のうち、

```text
複雑なモデルが常に勝つとは限らない
```

理由を2つ挙げてください。

---

## 問3

次の結果。

```text
Model A
MAE = 40

Model B
MAE = 50
```

どちらが良い？

---

## 問4

CVの結果。

```text
Model A
mean = 40
std = 1

Model B
mean = 39
std = 12
```

この結果から考えられることは？

---

## 問5

なぜ、

```text
Testデータ
```

を使って何度もハイパーパラメータ調整してはいけない？

---

# 🧪 今日の最終実習

## 回帰モデル比較パイプライン

今日の完成形。

```python
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression,
    Ridge
)
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

import pandas as pd
```

```python
data = load_diabetes()

X = data.data
y = data.target
```

モデル。

```python
models = {
    "Baseline": DummyRegressor(
        strategy="mean"
    ),

    "Linear": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}
```

CV。

```python
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

評価。

```python
results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    mae = -scores

    results.append({
        "model": name,
        "mae_mean": mae.mean(),
        "mae_std": mae.std()
    })
```

```python
results_df = pd.DataFrame(
    results
)

print(
    results_df.sort_values(
        "mae_mean"
    )
)
```

これで、

> **Baseline → 複数モデル → CV → 平均と分散 → 比較**

という、基本的な実験設計が完成。

---

# 🌱 今日のまとめ

今日の核心は、

> **モデル比較は「モデルを並べて数字を見る作業」ではない。**

ちゃんとした実験には、

```text
① Baseline

② 同じデータ

③ 同じ評価指標

④ 同じCV

⑤ 平均性能

⑥ ばらつき

⑦ モデルの複雑さ

⑧ 計算コスト

⑨ 説明可能性
```

を見る。

つまり、

```text
最強モデル
```

を探すのではなく、

> **その問題に対して、必要な性能と性質を満たすモデルを選ぶ。**

これが工学的なモデル選択。

---

# 🧭 AI工学101・現在地

ここまでで、

```text
データ
↓
特徴量
↓
前処理
↓
Pipeline
↓
複数モデル
↓
Cross Validation
↓
評価
↓
モデル選択
```

という、

> **小さな機械学習実験を一通り設計する能力**

がかなり形になってきた。

今日の内容は、PyTorchに入ってからもそのまま使う。

ニューラルネットでも、

```text
ベースライン
評価
CV
過学習
ハイパーパラメータ
モデル比較
```

は全部同じだからな。

---

# 🔜 第38回

## 分類モデルを本気で比較する：Logistic Regression → Random Forest → Gradient Boosting

次は分類版。

扱うのは、

* Logistic Regression
* Random Forest
* Gradient Boosting
* Accuracyだけではない評価
* Precision / Recall / F1
* ROC-AUC
* PR-AUC
* クラス不均衡
* 閾値調整
* Cross Validationによる公平な比較

テーマは、

> **「モデルの性能」は単一の数字ではなく、何を失敗として重く見るかによって変わる。**

次回は、分類モデルをガチで比較するぞ。🔥🧠🔧
ふふふ、ベンチマーク芸じゃなくて**実験設計**の時間だwwwww